```
预处理好的GSM8K tokenized数据集
        ↓
随机抽一个 micro-batch
        ↓
模型前向，拿到 log_probs / entropy / loss
        ↓
只对有效 response token 做 masked normalize
        ↓
梯度累积
        ↓
optimizer.step() 更新参数
        ↓
定期把 policy 权重同步到 vLLM 推理引擎
        ↓
用 vLLM 跑 GSM8K 验证集批量生成
        ↓
之前写的规则奖励函数打分
        ↓
WandB 记录训练与评测曲线
```
## 3.1 迭代
| 中文名 | 英文 | 以 GSM8K 训练为例 |
|--------|------|-------------------------------------|
| **物理批次** | micro-batch | 显卡一次前向传播吃进去的样本数，比方说 = 2（假设 16G 显存只能装下 2 条长 CoT）。 |
| **逻辑批次** | logical batch | 模型参数**真正更新一次**前，累计看到了多少样本。2 × 16 = 32。 |
| **梯度累积步** | gradient accumulation steps | 攒几次物理批次才做一次参数更新，这里 = 16。 |
| **优化器步** | step / optimizer step | 一次参数更新。一个 step 里包含 16 次前向+反向，然后一次 optimizer.step()。 |
| **轮次** | epoch | 数据集被完整遍历一次。但因为用的是 **随机抽样**（每次从总数据里随机抽一批），epoch 不再严格，变成“累计见过的样本数 ≈ 数据集大小”。 |
| **回合** | episode | 这个在 **监督学习** 里基本不用，是强化学习里的概念：从初始状态开始，到终止状态结束的一段完整交互轨迹。 |

在[隔壁](../huggingface/huggingface.ipynb)提到 **梯度累积** 下，epoch 已经不是最自然的控制单位了，step 才是。

如上表，一个逻辑批次之后，可能进行了n次反向传播，而模型真正更新一次参数，step++ ——

一个 step = 一次参数更新 = 指定多少个物理批次 × 梯度累积步。

> 用 `max_steps=1000` 比用 `epochs=3` 更直观，因为你不需要关心数据到底被看了几遍，只需要知道模型更新了多少次。
>
> Epoch 则更贴近“等效曝光量”，累计见过全量样本后，就算 1 个等效epoch。

## 3.2 Masked Normalize
我们不是在训练“整条 padded 序列上的平均交叉熵”，而是在训练 **只针对有效 response token 的平均交叉熵。**
否则分母里会混进：
* prompt token
* padding token
* 甚至一些本来不该负责的区域

这会造成两个很现实的问题：

1. **梯度被稀释**：短回答样本会被长 padding 样本拖平
2. **不同 batch 之间 loss 尺度不稳定**：因为每个 batch 的有效 token 数量都不一样


```
模型前向传播
    ↓
Logits (模型原始输出，形状 [B, L, V]，V是词表大小)
    ↓ (用 log_softmax 转换)
Log_Probs (对数概率，形状 [B, L, V]，每个token在词表上的对数概率)
    ↓ (用 gather 提取目标token的对数概率)
Target_Log_Prob (目标token的对数概率，形状 [B, L])
    ↓ (取负数，因为我们要最小化负对数似然)
Token_Level_Loss (逐token损失，形状 [B, L])
    ↓ (Masked Normalize：只对有效response token求和、再平均)
Final_Loss (最终归一化损失，标量，用于反向传播)
```

Logits 是模型最后一层线性层的**原始输出**，没有经过任何归一化，它的取值范围是 $(-\infty, +\infty)$，没有直接的概率意义。
- 用 `softmax(logits)` 可以把 logits 转成概率 $p_i \in [0,1]$，所有token的概率加起来等于1；
- 用 `log_softmax(logits)` 可以把 logits 转成**对数概率** $\log(p_i) \in (-\infty, 0]$，概率越接近1，对数概率越接近0；概率越接近0，对数概率越接近负无穷。

我们的训练目标是：**让模型对正确token的预测概率越大越好**。
- 如果模型对正确token的预测概率是 0.9（非常自信），对数概率是 $\log(0.9) \approx -0.1$，取负数后 Loss = 0.1（很小，模型很对）；
- 如果模型对正确token的预测概率是 0.1（非常迷茫），对数概率是 $\log(0.1) \approx -2.3$，取负数后 Loss = 2.3（很大，模型很错）。

这就是**负对数似然损失（Negative Log-Likelihood, NLL）**，也是交叉熵损失在单目标情况下的等价形式。

设每个位置的 token-level loss 为 $\ell_{b,t}$，
响应掩码为 $m_{b,t} \in \{0,1\}$。

那么一个 batch 的 masked mean loss 应该写成：

$$
\mathcal{L}_{\text{batch}}
=
\frac{\sum_{b,t} m_{b,t} \, \ell_{b,t}}
{\sum_{b,t} m_{b,t}}
$$

这个分母：

$$
\text{normalize\_constant} = \sum_{b,t} m_{b,t}
$$

In [ ]:
import torch
import torch.nn.functional as F

def masked_response_ce_loss(logits, labels, response_mask, ignore_index=-100):
    """
    logits: [B, L, V]
    labels: [B, L]
    response_mask: [B, L] 1 for response tokens, 0 for prompt
    """
    vocab_size = logits.size(-1)
    # 先逐token ce
    token_loss = F.cross_entropy(
        logits.view(-1, vocab_size),    # (batch_size * seq_len, vocab_size
        labels.view(-1),                # (batch_size * seq_len
                                        # -1 的意思是“你帮我自动算一下这一维应该是多少”，不加其他参数就是拉平成一维咯
        ignore_index=ignore_index,
        reduction='none',
        # reduction='mean'（默认）：所有位置的损失求平均，返回一个标量。
        # reduction='sum'：所有位置的损失求和。
        # reduction='none'：不合并，每个位置各自保留一个损失值。
    ).view_as(labels)
    # 先排除padding位置
    valid_mask = (labels != ignore_index).long()  # [B, L]
    final_mask = valid_mask * response_mask.long()  # [B, L] 只保留response部分

    # 计算最终损失
    numerator = (token_loss * final_mask).sum()  # 只累加response部分的损失
    denominator = final_mask.sum().clamp_min(1)  # response token总数，至少为1防除零
    loss = numerator / denominator
    return loss

In [1]:
import os
import re
import json
import math
import random
from fractions import Fraction
from typing import List, Dict, Optional, Tuple, Any

import numpy as np
import torch
import torch.nn.functional as F
import wandb

from tqdm.auto import tqdm
from torch.optim import AdamW
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    PreTrainedTokenizer,
    PreTrainedModel,
)

from vllm import LLM, SamplingParams
from unittest.mock import patch

# 统一约定标签常量
THINK_OPEN = "<think>"
THINK_CLOSE = "</think>"
ANSWER_OPEN = "<answer>"
ANSWER_CLOSE = "</answer>"

d:\Anaconda\envs\py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

## 3.3 训练主循环
## 3.3.1 初始化 vLLM

In [ ]:
def init_vllm(model_id: str, device: str, seed: int, gpu_memory_utilization: float):
    """
    初始化 vLLM 推理实例。
    """
    with patch("torch.distributed.get_world_size", return_value=1), \
         patch("vllm.worker.worker.Worker._assert_memory_footprint_increased_during_profiling", return_value=None):
        return LLM(
            model=model_id,
            device=device,
            dtype=torch.bfloat16,
            enable_prefix_caching=True,
            gpu_memory_utilization=gpu_memory_utilization,
            seed=seed,
        )

把训练中的 policy 权重同步到 vLLM：

In [ ]:
@torch.no_grad()
def load_policy_into_vllm_instance(policy: PreTrainedModel, llm: LLM):
    """
    将 policy 当前参数同步到 vLLM 实例中。
    """
    state_dict = {k: v.detach() for k, v in policy.state_dict().items()}

    llm_model = llm.llm_engine.model_executor.driver_worker.model_runner.model
    llm_model.load_weights(state_dict.items())

    print("\n[Sync] Policy weights synced to vLLM.")

## 3.3.2 辅助函数
从预处理好的 tensor 池里随机取一个 micro-batch：

In [ ]:
def get_batch(tokenized_data: Dict[str, torch.Tensor], batch_size: int, device: str):
    """
    从预处理好的 tokenized_data 中随机采样一个 micro-batch。
    支持 batch_size > 数据集长度时有放回采样。
    """
    total_len = tokenized_data["input_ids"].size(0)

    if batch_size <= total_len:
        batch_indices = random.sample(range(total_len), batch_size)
    else:
        batch_indices = random.choices(range(total_len), k=batch_size)

    idx = torch.tensor(batch_indices, dtype=torch.long)

    return {
        k: v[idx].to(device)
        for k, v in tokenized_data.items()
    }

一次性 tokenize 全部训练样本，并构造 response_mask / valid_mask：

In [ ]:
def tokenize_prompt_and_output(
    prompt_strs: List[str],
    output_strs: List[str],
    tokenizer: PreTrainedTokenizer,
    add_eos_to_output: bool = True,
) -> Dict[str, torch.Tensor]:
    """
    对 prompt 和 response 进行分词、拼接、padding、shift，
    返回：
        - input_ids
        - labels
        - response_mask   (与 labels 对齐)
        - valid_mask      (与 labels 对齐，1 表示非 padding 的真实 label)
        - attention_mask  (与 input_ids 对齐)
    """
    all_combined_ids = []
    all_response_masks = []
    all_valid_masks = []

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    if pad_id is None:
        raise ValueError("tokenizer 必须至少有 pad_token_id 或 eos_token_id 之一。")

    for p_str, o_str in zip(prompt_strs, output_strs):
        p_ids = tokenizer.encode(p_str, add_special_tokens=False)
        o_ids = tokenizer.encode(o_str, add_special_tokens=False)

        if add_eos_to_output and tokenizer.eos_token_id is not None:
            if len(o_ids) == 0 or o_ids[-1] != tokenizer.eos_token_id:
                o_ids = o_ids + [tokenizer.eos_token_id]

        combined_ids = p_ids + o_ids
        response_mask = [0] * len(p_ids) + [1] * len(o_ids)
        valid_mask = [1] * len(combined_ids)

        all_combined_ids.append(combined_ids)
        all_response_masks.append(response_mask)
        all_valid_masks.append(valid_mask)

    max_len = max(len(x) for x in all_combined_ids)
    batch_size = len(all_combined_ids)

    padded_inputs = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    padded_response_masks = torch.zeros((batch_size, max_len), dtype=torch.long)
    padded_valid_masks = torch.zeros((batch_size, max_len), dtype=torch.long)

    for i, (ids, r_mask, v_mask) in enumerate(zip(all_combined_ids, all_response_masks, all_valid_masks)):
        L = len(ids)
        padded_inputs[i, :L] = torch.tensor(ids, dtype=torch.long)
        padded_response_masks[i, :L] = torch.tensor(r_mask, dtype=torch.long)
        padded_valid_masks[i, :L] = torch.tensor(v_mask, dtype=torch.long)

    # Shift
    final_input_ids = padded_inputs[:, :-1].contiguous()
    final_labels = padded_inputs[:, 1:].clone().contiguous()

    final_response_mask = padded_response_masks[:, 1:].contiguous()
    final_valid_mask = padded_valid_masks[:, 1:].contiguous()

    # attention_mask 对应 input_ids 位置
    final_attention_mask = padded_valid_masks[:, :-1].contiguous()

    return {
        "input_ids": final_input_ids,
        "labels": final_labels,
        "response_mask": final_response_mask,
        "valid_mask": final_valid_mask,
        "attention_mask": final_attention_mask,
    }

token 熵计算：

In [ ]:
def compute_entropy(logits: torch.Tensor) -> torch.Tensor:
    """
    计算每个位置 next-token 分布的熵。
    logits: [B, L, V]
    返回:   [B, L]
    """
    lse = torch.logsumexp(logits, dim=-1)              # [B, L]
    probs = F.softmax(logits, dim=-1)                  # [B, L, V]
    exp_logits = torch.sum(probs * logits, dim=-1)     # [B, L]
    entropy = lse - exp_logits
    return entropy

拿到每个位置上真实 label 的 log-prob，以及可选的 token entropy：

In [ ]:
def get_response_log_probs(
    model: PreTrainedModel,
    input_ids: torch.Tensor,
    labels: torch.Tensor,
    attention_mask: Optional[torch.Tensor] = None,
    return_token_entropy: bool = False,
) -> Dict[str, torch.Tensor]:
    """
    获取模型对 labels 的条件对数概率。
    这里默认 input_ids / labels 已经在数据预处理时做过 shift。
    """
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )
    logits = outputs.logits  # [B, L, V]

    log_probs_all = F.log_softmax(logits, dim=-1)

    safe_labels = labels.clamp_min(0)
    log_probs = torch.gather(
        log_probs_all,
        dim=-1,
        index=safe_labels.unsqueeze(-1)
    ).squeeze(-1)  # [B, L]

    result = {"log_probs": log_probs}

    if return_token_entropy:
        result["token_entropy"] = compute_entropy(logits)

    return result

单个 micro-batch 的 SFT 损失（直接在 log_probs 层做 masked NLL，会更自然）：

In [ ]:
def sft_microbatch_train_step(
    policy_log_probs: torch.Tensor,
    response_mask: torch.Tensor,
    gradient_accumulation_steps: int,
    normalize_constant: Optional[float] = None,
):
    """
    基于已提取出的 policy_log_probs，计算 response-only 的 masked NLL。
    """
    token_nll = -policy_log_probs
    mask = response_mask.float()

    if normalize_constant is None:
        normalize_constant = mask.sum().clamp_min(1.0)
    else:
        if not torch.is_tensor(normalize_constant):
            normalize_constant = torch.tensor(
                float(normalize_constant),
                device=policy_log_probs.device,
                dtype=policy_log_probs.dtype,
            )

    numerator = (token_nll * mask).sum()
    loss = numerator / normalize_constant
    loss = loss / gradient_accumulation_steps

    stats = {
        "masked_nll_sum": numerator.detach().item(),
        "num_response_tokens": mask.sum().detach().item(),
    }
    return loss, stats

### 3.3.2 奖励函数需要的字符串解析 helper

In [ ]:
def extract_tag_text(text: str, open_tag: str, close_tag: str) -> Optional[str]:
    pattern = re.escape(open_tag) + r"(.*?)" + re.escape(close_tag)
    m = re.search(pattern, text, flags=re.DOTALL)
    return m.group(1).strip() if m else None


def extract_boxed_answer(text: str) -> Optional[str]:
    """
    提取 \boxed{...} 中的内容。
    这里只做 GSM8K 风格的轻量解析。
    """
    m = re.search(r"\\boxed\{([^{}]+)\}", text)
    return m.group(1).strip() if m else None


def normalize_gsm8k_answer(ans: str) -> str:
    if ans is None:
        return ""
    ans = str(ans).strip()
    ans = ans.replace(",", "")
    ans = ans.replace("$", "")
    ans = ans.replace(" ", "")
    ans = ans.replace("\n", "")
    return ans


def try_parse_number(ans: str) -> Optional[float]:
    ans = normalize_gsm8k_answer(ans)

    # 先试分数
    if "/" in ans:
        try:
            return float(Fraction(ans))
        except Exception:
            pass

    # 再试浮点
    try:
        return float(ans)
    except Exception:
        return None


def answers_match(pred: str, gold: str, rel_tol: float = 1e-4) -> bool:
    pred_norm = normalize_gsm8k_answer(pred)
    gold_norm = normalize_gsm8k_answer(gold)

    if pred_norm == gold_norm:
        return True

    pred_num = try_parse_number(pred_norm)
    gold_num = try_parse_number(gold_norm)

    if pred_num is not None and gold_num is not None:
        return math.isclose(pred_num, gold_num, rel_tol=rel_tol)

    return False

### 3.3.3 奖励函数
没有 think/answer 标签就直接 0 分。

In [ ]:
def r1_zero_reward_fn(response: str, ground_truth, fast: bool = True):
    """
    严格格式奖励：
      - 必须包含 think / answer 标签
      - 只从 <answer> ... </answer> 中抽答案
      - GSM8K 风格答案用数值/字符串规则比对
    """
    think_text = extract_tag_text(response, THINK_OPEN, THINK_CLOSE)
    answer_text = extract_tag_text(response, ANSWER_OPEN, ANSWER_CLOSE)

    format_ok = (think_text is not None) and (answer_text is not None)

    if not format_ok:
        return {
            "format_reward": 0.0,
            "answer_reward": 0.0,
            "reward": 0.0,
            "is_correct": False,
            "model_answer": None,
        }

    # 若 answer span 里还有 \boxed{...}，优先取 boxed 内容
    if "\\boxed{" in answer_text:
        boxed = extract_boxed_answer(answer_text)
        if boxed is not None:
            answer_text = boxed

    if isinstance(ground_truth, list):
        is_correct = any(answers_match(answer_text, gt) for gt in ground_truth)
    else:
        is_correct = answers_match(answer_text, ground_truth)

    return {
        "format_reward": 1.0,
        "answer_reward": 1.0 if is_correct else 0.0,
        "reward": 1.0 if is_correct else 0.0,
        "is_correct": bool(is_correct),
        "model_answer": answer_text,
    }

### 3.3.4 vLLM 验证集生成 + 打分 + 记录指标

In [ ]:
def log_generations(
    vllm_model: LLM,
    sampling_params: SamplingParams,
    prompts: List[str],
    ground_truths: List[str],
    reward_fn,
    step: int,
    log_prefix: str = "eval",
):
    """
    用 vLLM 生成验证集答案，调用 reward_fn 打分，并写入 wandb。
    """
    outputs = vllm_model.generate(prompts, sampling_params)

    format_scores = []
    answer_scores = []
    rewards = []
    is_corrects = []
    lengths = []
    lengths_correct = []
    lengths_incorrect = []

    rows = []

    for prompt, gt, out in zip(prompts, ground_truths, outputs):
        if len(out.outputs) == 0:
            gen_text = ""
            gen_len = 0
        else:
            first = out.outputs[0]
            gen_text = first.text
            if hasattr(first, "token_ids") and first.token_ids is not None:
                gen_len = len(first.token_ids)
            else:
                gen_len = len(gen_text)

        reward_info = reward_fn(gen_text, gt)

        format_scores.append(reward_info["format_reward"])
        answer_scores.append(reward_info["answer_reward"])
        rewards.append(reward_info["reward"])
        is_corrects.append(int(reward_info["is_correct"]))
        lengths.append(gen_len)

        if reward_info["is_correct"]:
            lengths_correct.append(gen_len)
        else:
            lengths_incorrect.append(gen_len)

        rows.append({
            "prompt": prompt,
            "ground_truth": gt,
            "output": gen_text,
            "parsed_answer": reward_info.get("model_answer", None),
            "is_correct": reward_info["is_correct"],
        })

    metrics = {
        f"{log_prefix}/accuracy": float(np.mean(is_corrects)) if len(is_corrects) else 0.0,
        f"{log_prefix}/format_score": float(np.mean(format_scores)) if len(format_scores) else 0.0,
        f"{log_prefix}/answer_score": float(np.mean(answer_scores)) if len(answer_scores) else 0.0,
        f"{log_prefix}/avg_reward": float(np.mean(rewards)) if len(rewards) else 0.0,
        f"{log_prefix}/avg_length": float(np.mean(lengths)) if len(lengths) else 0.0,
        f"{log_prefix}/avg_length_correct": float(np.mean(lengths_correct)) if len(lengths_correct) else 0.0,
        f"{log_prefix}/avg_length_incorrect": float(np.mean(lengths_incorrect)) if len(lengths_incorrect) else 0.0,
        "eval_step": step,
    }

    wandb.log(metrics)
    return metrics

### 3.3.5 环境初始化
- 根据 batch_size 和 micro_batch_size 自动计算梯度累积步数 grad_accum_steps。
- 初始化 wandb 项目，并加载 prompt 模板（用于后面验证集格式化）。

In [ ]:
def init_experiment_config(
    batch_size,
    micro_batch_size,
    wandb_project,
    wandb_run_name,
    prompt_path,
    extra_config=None,
):
    """
    计算累积步数，初始化 wandb，加载 prompt 模板。
    返回 grad_accum_steps, wandb_run, r1_template
    """
    if batch_size % micro_batch_size != 0:
        raise ValueError("batch_size 必须能被 micro_batch_size 整除。")

    grad_accum_steps = batch_size // micro_batch_size

    config = dict(extra_config or {})
    config["grad_accum_steps"] = grad_accum_steps

    run = wandb.init(
        project=wandb_project,
        name=wandb_run_name,
        config=config,
    )

    # 显式绑定 x 轴
    run.define_metric("train_step")
    run.define_metric("train/*", step_metric="train_step")
    run.define_metric("eval_step")
    run.define_metric("eval/*", step_metric="eval_step")

    with open(prompt_path, "r") as f:
        r1_template = f.read().strip()

    return grad_accum_steps, run, r1_template

### 3.3.6 模型与分词器初始化
- 加载 tokenizer，补齐 pad_token。
- 训练 policy 模型用 FlashAttention-2 加速，放到 args.device（cuda:0）。
- 开启梯度检查点以节省显存，并初始化 AdamW 优化器。
- 调用 init_vllm 在 args.vllm_device（cuda:1）上初始化独立的推理引擎，注意 enable_prefix_caching=True。

In [ ]:
def load_models_and_optimizer(model_id, device, vllm_device, seed, vllm_gpu_util, lr):
    """
    加载 tokenizer, 训练 policy, optimizer, vLLM 推理实例。
    返回 tokenizer, policy, optimizer, vllm_inst
    """
    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 训练模型
    policy = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        attn_implementation="flash_attention_2",
    ).to(device)

    # gradient checkpointing 时，训练态通常关闭 use_cache
    policy.config.use_cache = False
    policy.gradient_checkpointing_enable()
    policy.train()

    optimizer = AdamW(policy.parameters(), lr=lr)

    # vLLM 推理实例
    vllm_inst = init_vllm(model_id, vllm_device, seed, vllm_gpu_util)

    return tokenizer, policy, optimizer, vllm_inst

### 3.3.7 训练集预处理


In [ ]:
def prepare_training_data(train_data_path, tokenizer, filter_correct=False, dataset_size=None):
    """
    加载并 tokenize 整个训练集。
    返回 tokenized_train_data 字典：
      input_ids, labels, response_mask, valid_mask, attention_mask
    """
    raw_train_data = []
    with open(train_data_path, "r") as f:
        for line in f:
            raw_train_data.append(json.loads(line))

    if filter_correct:
        raw_train_data = [item for item in raw_train_data if item.get("is_correct", True)]

    if dataset_size and dataset_size < len(raw_train_data):
        raw_train_data = random.sample(raw_train_data, dataset_size)

    tokenized_train_data = tokenize_prompt_and_output(
        prompt_strs=[item["prompt"] for item in raw_train_data],
        output_strs=[item["response"] for item in raw_train_data],
        tokenizer=tokenizer,
    )
    return tokenized_train_data

### 3.3.8 验证集准备

In [ ]:
def prepare_validation_data(val_data_path, prompt_template, max_eval_samples, max_tokens):
    """
    加载验证集，生成 prompt 列表和 ground truth 列表。
    返回 val_prompts, val_ground_truths, eval_sampling_params
    """
    val_prompts = []
    val_ground_truths = []

    with open(val_data_path, "r") as f:
        for i, line in enumerate(f):
            if i >= max_eval_samples:
                break

            item = json.loads(line)
            raw_a = item["answer"]
            gold = raw_a.split("####")[-1].strip() if "####" in raw_a else raw_a.strip()

            formatted_prompt = prompt_template.replace("{question}", item["question"])
            val_prompts.append(formatted_prompt)
            val_ground_truths.append(gold)

    eval_sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=max_tokens,
        stop=[ANSWER_CLOSE],
        include_stop_str_in_output=True,
    )

    return val_prompts, val_ground_truths, eval_sampling_params

### 3.3.9 Step 0 基线评估

In [ ]:
def run_step0_evaluation(policy, vllm_inst, eval_sampling_params, val_prompts, val_ground_truths):
    """
    Step 0 评估，返回评估指标字典。
    """
    print("\n[Step 0] Starting Evaluation...")
    policy.eval()
    load_policy_into_vllm_instance(policy, vllm_inst)

    metrics = log_generations(
        vllm_model=vllm_inst,
        sampling_params=eval_sampling_params,
        prompts=val_prompts,
        ground_truths=val_ground_truths,
        reward_fn=r1_zero_reward_fn,
        step=0,
        log_prefix="eval",
    )
    print(f"Eval Accuracy: {metrics.get('eval/accuracy', 0):.2%}")

    policy.train()
    return metrics

### 3.3.10 单个训练step（含梯度累积）


In [ ]:
def train_one_step(
    policy,
    optimizer,
    tokenized_train_data,
    micro_batch_size,
    grad_accum_steps,
    tokenizer,
    device,
    step,
):
    """
    执行一次 optimizer.step()。
    返回：
      avg_loss, avg_global_entropy, avg_response_entropy
    """
    policy.train()

    accumulated_loss = 0.0
    accumulated_entropy = 0.0
    accumulated_res_entropy = 0.0

    for _ in range(grad_accum_steps):
        batch = get_batch(tokenized_train_data, micro_batch_size, device)

        response_outputs = get_response_log_probs(
            model=policy,
            input_ids=batch["input_ids"],
            labels=batch["labels"],
            attention_mask=batch["attention_mask"],
            return_token_entropy=True,
        )

        log_probs = response_outputs["log_probs"]
        token_entropy = response_outputs["token_entropy"]

        with torch.no_grad():
            valid_token_mask = batch["valid_mask"].bool()
            current_res_mask = batch["response_mask"].bool() & valid_token_mask

            avg_res_entropy = (
                token_entropy[current_res_mask].mean().item()
                if current_res_mask.any() else 0.0
            )
            avg_global_entropy = (
                token_entropy[valid_token_mask].mean().item()
                if valid_token_mask.any() else 0.0
            )

            normalize_constant = current_res_mask.sum().clamp_min(1).item()

        loss, _ = sft_microbatch_train_step(
            policy_log_probs=log_probs,
            response_mask=current_res_mask,
            gradient_accumulation_steps=grad_accum_steps,
            normalize_constant=normalize_constant,
        )
        loss.backward()

        # 还原为“未除 accum steps 前”的尺度，便于日志直观理解
        accumulated_loss += loss.item() * grad_accum_steps
        accumulated_entropy += avg_global_entropy
        accumulated_res_entropy += avg_res_entropy

    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    avg_loss = accumulated_loss / grad_accum_steps
    avg_entropy = accumulated_entropy / grad_accum_steps
    avg_res_entropy = accumulated_res_entropy / grad_accum_steps

    wandb.log({
        "train/loss": avg_loss,
        "train/global_entropy": avg_entropy,
        "train/response_entropy": avg_res_entropy,
        "train_step": step + 1,
    })

    return avg_loss, avg_entropy, avg_res_entropy

### 3.3.11 定期评估与权重同步

In [ ]:
def evaluate_and_sync(
    policy,
    vllm_inst,
    eval_sampling_params,
    val_prompts,
    val_ground_truths,
    step,
):
    """
    同步权重、评估、记录，返回评估指标字典。
    """
    print(f"\n[Step {step}] Starting Evaluation...")
    policy.eval()
    load_policy_into_vllm_instance(policy, vllm_inst)

    metrics = log_generations(
        vllm_model=vllm_inst,
        sampling_params=eval_sampling_params,
        prompts=val_prompts,
        ground_truths=val_ground_truths,
        reward_fn=r1_zero_reward_fn,
        step=step,
        log_prefix="eval",
    )
    print(f"Eval Accuracy: {metrics.get('eval/accuracy', 0):.2%}")

    policy.train()
    return metrics

### 3.3.12 模型保存

In [ ]:
def save_final_model(policy, tokenizer, output_dir, max_steps, dataset_size, filter_correct):
    """
    保存最终的 model + tokenizer。
    """
    save_name = f"sft_steps_{max_steps}_subset{dataset_size}_filtered{filter_correct}"
    save_path = os.path.join(output_dir, save_name)
    os.makedirs(save_path, exist_ok=True)

    policy.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    print(f"Model saved to {save_path}")

### 3.3.13 主训练循环

In [ ]:
def run_sft_experiment(
    model_id,
    train_data_path,
    val_data_path,
    prompt_path,
    output_dir,
    batch_size,
    micro_batch_size,
    max_steps,
    lr,
    seed,
    max_tokens,
    dataset_size=None,
    filter_correct=False,
    device="cuda:0",
    vllm_device="cuda:1",
    vllm_gpu_util=0.5,
    eval_every_steps=20,
    max_eval_samples=100,
    wandb_project="cs336-sft",
    wandb_run_name=None,
):
    config_dict = {
        "model_id": model_id,
        "train_data_path": train_data_path,
        "val_data_path": val_data_path,
        "prompt_path": prompt_path,
        "output_dir": output_dir,
        "batch_size": batch_size,
        "micro_batch_size": micro_batch_size,
        "max_steps": max_steps,
        "lr": lr,
        "seed": seed,
        "max_tokens": max_tokens,
        "dataset_size": dataset_size,
        "filter_correct": filter_correct,
        "device": device,
        "vllm_device": vllm_device,
        "vllm_gpu_util": vllm_gpu_util,
        "eval_every_steps": eval_every_steps,
        "max_eval_samples": max_eval_samples,
        "wandb_project": wandb_project,
        "wandb_run_name": wandb_run_name,
    }

    # 1. 配置 & wandb
    grad_accum_steps, run, r1_template = init_experiment_config(
        batch_size=batch_size,
        micro_batch_size=micro_batch_size,
        wandb_project=wandb_project,
        wandb_run_name=wandb_run_name,
        prompt_path=prompt_path,
        extra_config=config_dict,
    )

    # 2. 模型
    tokenizer, policy, optimizer, vllm_inst = load_models_and_optimizer(
        model_id=model_id,
        device=device,
        vllm_device=vllm_device,
        seed=seed,
        vllm_gpu_util=vllm_gpu_util,
        lr=lr,
    )

    optimizer.zero_grad(set_to_none=True)

    # 3. 训练集
    tokenized_train_data = prepare_training_data(
        train_data_path=train_data_path,
        tokenizer=tokenizer,
        filter_correct=filter_correct,
        dataset_size=dataset_size,
    )
    print(f"Training samples: {len(tokenized_train_data['input_ids'])}")

    # 4. 验证集
    val_prompts, val_ground_truths, eval_sampling_params = prepare_validation_data(
        val_data_path=val_data_path,
        prompt_template=r1_template,
        max_eval_samples=max_eval_samples,
        max_tokens=max_tokens,
    )

    # 5. Step 0
    run_step0_evaluation(
        policy=policy,
        vllm_inst=vllm_inst,
        eval_sampling_params=eval_sampling_params,
        val_prompts=val_prompts,
        val_ground_truths=val_ground_truths,
    )

    # 6. 训练循环
    progress_bar = tqdm(range(max_steps), desc="SFT Steps")

    for step in range(max_steps):
        train_one_step(
            policy=policy,
            optimizer=optimizer,
            tokenized_train_data=tokenized_train_data,
            micro_batch_size=micro_batch_size,
            grad_accum_steps=grad_accum_steps,
            tokenizer=tokenizer,
            device=device,
            step=step,
        )
        progress_bar.update(1)

        if (step + 1) % eval_every_steps == 0:
            evaluate_and_sync(
                policy=policy,
                vllm_inst=vllm_inst,
                eval_sampling_params=eval_sampling_params,
                val_prompts=val_prompts,
                val_ground_truths=val_ground_truths,
                step=step + 1,
            )

    # 7. 保存
    save_final_model(
        policy=policy,
        tokenizer=tokenizer,
        output_dir=output_dir,
        max_steps=max_steps,
        dataset_size=dataset_size,
        filter_correct=filter_correct,
    )

    wandb.finish()

    return {
        "policy": policy,
        "tokenizer": tokenizer,
        "vllm_inst": vllm_inst,
        "grad_accum_steps": grad_accum_steps,
    }

启动训练！

In [ ]:
run_sft_experiment(
    model_id="model/Qwen2.5-Math-1.5B",
    train_data_path="data/gsm8k/train_sft_reason_gsm8k_raw.jsonl",
    val_data_path="data/gsm8k/test.jsonl",
    prompt_path="cs336_alignment/prompts/r1_zero.prompt",
    output_dir="result/checkpoints",
    batch_size=16,
    micro_batch_size=1,
    max_steps=200,
    lr=2e-5,
    seed=42,
    max_tokens=1024,
    dataset_size=128,
    filter_correct=False,
    device="cuda:0",
    vllm_device="cuda:1",
    vllm_gpu_util=0.5,
    eval_every_steps=20,
    max_eval_samples=100,
    wandb_project="cs336-sft",
    wandb_run_name="demo-run",
)